In [16]:
import os
import pandas as pd
import numpy as np
import oracledb

caminho_pasta = 'C:/Users/Paulo/Documents/FIAP/GLOBAL_SOLUTION_2026_01/DADOS_CLIMA'
lista_colunas = [f'Coluna_{i}' for i in range(26)]

# Colunas numéricas que precisam de limpeza de texto
colunas_numericas = [
    'PRECIPITACAO_mm', 'TEMPERATURA_C', 'TEMPERATURA_MAX_C', 'TEMPERATURA_MIN_C',
    'UMIDADE_MAX', 'UMIDADE_MIN', 'UMIDADE_RELATIVA', 
    'VENTO_HORARIO', 'VENTO_RAJADA_MAX', 'VENTO_VELOCIDADE'
]

colunas_agrupamento = [
    'REGIAO', 'UF', 'ESTACAO', 'COD', 
    'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'DATA_FUNDACAO', 'ANO_MES'
]

# Comando SQL de inserção
sql_insercao = """
INSERT INTO DADOS_CLIMA_GS_2026_S1 (
    REGIAO, UF, ESTACAO, COD, LATITUDE, LONGITUDE, ALTITUDE, DATA_FUNDACAO, ANO_MES,
    PRECIPITACAO_mm, TEMPERATURA_C, TEMPERATURA_MAX_C, TEMPERATURA_MIN_C,
    UMIDADE_MAX, UMIDADE_MIN, UMIDADE_RELATIVA, VENTO_HORARIO, VENTO_RAJADA_MAX, VENTO_VELOCIDADE
) VALUES (
    :1, :2, :3, :4, :5, :6, :7, :8, :9,
    :10, :11, :12, :13, :14, :15, :16, :17, :18, :19
)
"""

# Verifica se a pasta existe antes de iniciar
if not os.path.exists(caminho_pasta):
    print(f"Erro: A pasta '{caminho_pasta}' não existe.")
else:
    arquivos = [f for f in os.listdir(caminho_pasta) if f.lower().endswith('.csv')]
    print(f"Total de arquivos encontrados na pasta: {len(arquivos)}")

    # Abre a conexão com o banco antes do loop para evitar abrir/fechar toda hora
    try:
        conn = oracledb.connect(
            user='rm567787',
            password='281083',
            dsn='oracle.fiap.com.br:1521/ORCL'
        )
        cursor = conn.cursor()
        print("Conexão com o banco de dados estabelecida.")

        # Loop processando arquivo por arquivo
        for idx, nome_arquivo in enumerate(arquivos, start=1):
            caminho_completo = os.path.join(caminho_pasta, nome_arquivo)
            print(f"[{idx}/{len(arquivos)}] Processando: {nome_arquivo}")
            
            try:
                # 1. Leitura rápida do arquivo individual
                df = pd.read_csv(
                    caminho_completo, 
                    sep=';', 
                    encoding='latin-1',
                    names=lista_colunas, 
                    header=None
                )
                
                # 2. Extração dos metadados do cabeçalho
                REGIAO = df.iloc[0, 1]
                UF = df.iloc[1, 1]
                ESTACAO = df.iloc[2, 1]
                COD = df.iloc[3, 1]       
                LATITUDE = df.iloc[4, 1]
                LONGITUDE = df.iloc[5, 1]
                ALTITUDE = df.iloc[6, 1]
                DATA_FUNDACAO = df.iloc[7, 1] 
                
                # Corta as linhas de metadados
                df = df.iloc[9:].copy()
                
                # Renomeia as colunas necessárias
                df = df.rename(columns={
                    'Coluna_0': 'DATA', 'Coluna_1': 'HORA', 'Coluna_2': 'PRECIPITACAO_mm',
                    'Coluna_3': 'PRESSAO_ATM_ESTACAO_mB', 'Coluna_4': 'PRESSAO_ATM_MAX_AUT_mB',
                    'Coluna_5': 'PRESSAO_ATM_MIN_AUT_mB', 'Coluna_6': 'RADIACAO_kJM2',
                    'Coluna_7': 'TEMPERATURA_C', 'Coluna_8': 'TEMPERATURA_P_ORVALHO_C',
                    'Coluna_9': 'TEMPERATURA_MAX_C', 'Coluna_10': 'TEMPERATURA_MIN_C',
                    'Coluna_11': 'TEMPERATURA_ORVALHO_MAX_C', 'Coluna_12': 'TEMPERATURA_ORVALHO_MIN_C',
                    'Coluna_13': 'UMIDADE_MAX', 'Coluna_14': 'UMIDADE_MIN', 'Coluna_15': 'UMIDADE_RELATIVA',
                    'Coluna_16': 'VENTO_HORARIO', 'Coluna_17': 'VENTO_RAJADA_MAX', 'Coluna_18': 'VENTO_VELOCIDADE'
                })
                
                # Tratamento de datas e criação da coluna ANO_MES
                df['DATA'] = pd.to_datetime(df['DATA'], errors='coerce')
                df['ANO_MES'] = df['DATA'].dt.strftime('%y_%m')
                
                # Adiciona metadados como colunas
                df["REGIAO"] = REGIAO
                df["UF"] = UF
                df["ESTACAO"] = ESTACAO
                df["COD"] = COD
                df["LATITUDE"] = LATITUDE
                df["LONGITUDE"] = LONGITUDE
                df["ALTITUDE"] = ALTITUDE
                df["DATA_FUNDACAO"] = DATA_FUNDACAO

                # 3. Limpeza das colunas numéricas (Vírgula para Ponto)
                for col in colunas_numericas:
                    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
                    df[col] = pd.to_numeric(df[col], errors='coerce')

                # 4. Agrupamento local (apenas as 12 linhas mensais deste arquivo)
                df_resumo = df.groupby(colunas_agrupamento).agg({
                    'PRECIPITACAO_mm': 'sum',
                    'TEMPERATURA_C': 'mean',
                    'TEMPERATURA_MAX_C': 'mean',
                    'TEMPERATURA_MIN_C': 'mean',
                    'UMIDADE_MAX': 'mean',
                    'UMIDADE_MIN': 'mean',
                    'UMIDADE_RELATIVA': 'mean',
                    'VENTO_HORARIO': 'mean',
                    'VENTO_RAJADA_MAX': 'mean',
                    'VENTO_VELOCIDADE': 'mean'
                }).reset_index()

                # Reordena explicitamente para garantir o mapeamento exato do SQL (19 colunas)
                df_resumo = df_resumo[[
                    'REGIAO', 'UF', 'ESTACAO', 'COD', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'DATA_FUNDACAO', 'ANO_MES',
                    'PRECIPITACAO_mm', 'TEMPERATURA_C', 'TEMPERATURA_MAX_C', 'TEMPERATURA_MIN_C',
                    'UMIDADE_MAX', 'UMIDADE_MIN', 'UMIDADE_RELATIVA', 'VENTO_HORARIO', 'VENTO_RAJADA_MAX', 'VENTO_VELOCIDADE'
                ]]

                # 5. Preparar dados para o Oracle (troca NaN por None)
                df_limpo = df_resumo.replace({np.nan: None})
                dados_para_inserir = df_limpo.values.tolist()

                # 6. Inserção em lote do arquivo atual
                if dados_para_inserir:
                    cursor.executemany(sql_insercao, dados_para_inserir)
                    conn.commit()  # Confirma o lote deste arquivo no banco
                    print(f"-> Sucesso: {len(dados_para_inserir)} linhas salvas no banco.")

            except Exception as e:
                print(f"-> Erro ao processar o arquivo {nome_arquivo}: {e}")
                conn.rollback()

    except oracledb.DatabaseError as e:
        erro, = e.args
        print(f"Erro de conexão com o banco de dados: {erro.message}")
        
    finally:
        # Garante o fechamento correto das conexões ao finalizar tudo
        if 'cursor' in locals():
            cursor.close()
        if 'conn' in locals():
            conn.close()
        print("\nProcesso concluído e conexões encerradas.")


Total de arquivos encontrados na pasta: 10046
Conexão com o banco de dados estabelecida.
[1/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2001_A_31-12-2001.CSV
-> Sucesso: 12 linhas salvas no banco.
[2/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2002_A_31-12-2002.CSV
-> Sucesso: 12 linhas salvas no banco.
[3/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2003_A_31-12-2003.CSV
-> Sucesso: 12 linhas salvas no banco.
[4/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2004_A_31-12-2004.CSV
-> Sucesso: 12 linhas salvas no banco.
[5/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2005_A_31-12-2005.CSV
-> Sucesso: 12 linhas salvas no banco.
[6/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2006_A_31-12-2006.CSV
-> Sucesso: 12 linhas salvas no banco.
[7/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2007_A_31-12-2007.CSV
-> Sucesso: 12 linhas salvas no banco.
[8/10046] Processando: INMET_CO_DF_A001_BRASILIA_01-01-2008_A_31-12-2008.CSV
-> Sucesso: 12 linhas 

In [26]:
import pandas as pd
import numpy as np
import oracledb

# ==============================================================================
# 1. CAPTURA E TRATAMENTO DOS DADOS (NOAA)
# ==============================================================================
url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

# Ler os dados tratando múltiplos espaços vazios como delimitador
oni_data = pd.read_csv(url, delim_whitespace=True)

# Definição do tipo de evento (El Niño / La Niña / Neutro)
condicoes_evento = [
    oni_data['ANOM'] < -0.5,
    oni_data['ANOM'] > 0.5
]
valores_evento = ['LA_NINA', 'EL_NINO']
oni_data['tipo_evento'] = np.select(condicoes_evento, valores_evento, default='NEUTRO')

# Definição da intensidade do evento estacional
abs_anom = oni_data['ANOM'].abs()
condicoes_intensidade = [
    abs_anom < 0.5,
    abs_anom <= 0.9,
    abs_anom <= 1.4,
    abs_anom <= 2.0
]
valores_intensidade = ['0_NEUTRO', '1_FRACO', '2_MODERADO', '3_FORTE']
oni_data['evento_intensidade'] = np.select(condicoes_intensidade, valores_intensidade, default='4_MUITO_FORTE')

# Mapeamento de siglas de trimestres para strings numéricas de meses
mapeamento_seas = {
    'DJF': '01', 'JFM': '02', 'FMA': '03', 'MAM': '04',
    'AMJ': '05', 'MJJ': '06', 'JJA': '07', 'JAS': '08',
    'ASO': '09', 'SON': '10', 'OND': '11', 'NDJ': '12'
}
oni_data['MES'] = oni_data['SEAS'].map(mapeamento_seas)
oni_data['ANO'] = oni_data['YR'].astype(str).str[-2:]
oni_data['ANO_MES'] = oni_data['ANO'].astype(str) + '_' + oni_data['MES'].astype(str)

print("Amostra dos dados processados:")
print(oni_data.head(12))

# ==============================================================================
# 2. PREPARAÇÃO DOS DADOS PARA O ORACLE (CORREÇÃO DE COLUNAS E TIPOS)
# ==============================================================================
# CORREÇÃO: Filtrar apenas as colunas exatas do INSERT na ordem correta dos parâmetros (:1 a :9)
colunas_sql = ['SEAS', 'YR', 'TOTAL', 'ANOM', 'tipo_evento', 'evento_intensidade', 'MES', 'ANO', 'ANO_MES']
df_insert = oni_data[colunas_sql].copy()

# CORREÇÃO DE TIPOS: Converter tipos do NumPy/Pandas para tipos nativos do Python
# Isso evita falhas de driver (ex: 'Int64' do pandas vs 'int' do Python)
dados_para_inserir = [tuple(x) for x in df_insert.to_numpy()]

# ==============================================================================
# 3. CONEXÃO E BANCO DE DADOS (ORACLE)
# ==============================================================================
sql_limpeza = "TRUNCATE TABLE DADOS_NOAA"
sql_insercao = """
INSERT INTO DADOS_NOAA (
    SEAS, YR, TOTAL, ANOM, tipo_evento, evento_intensidade, MES, ANO, ANO_MES
) VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
"""

conn = None
cursor = None

try:
    conn = oracledb.connect(
        user='rm567787',
        password='281083',
        dsn='oracle.fiap.com.br:1521/ORCL'
    )
    cursor = conn.cursor()
    
    # Executa a limpeza da tabela
    cursor.execute(sql_limpeza)
    print("Tabela 'DADOS_NOAA' limpa com sucesso.")
    
    # Inserção em lote (Bulk Insert eficiente)
    cursor.executemany(sql_insercao, dados_para_inserir)
    conn.commit()
    print(f"Sucesso! {len(dados_para_inserir)} linhas inseridas na tabela 'DADOS_NOAA'.")

except Exception as e:
    print(f"Erro no processo Oracle: {e}")
    if conn:
        conn.rollback()

finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()


C:\Users\Paulo\AppData\Local\Temp\ipykernel_26992\3711512160.py:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  oni_data = pd.read_csv(url, delim_whitespace=True)


Amostra dos dados processados:
   SEAS    YR  TOTAL  ANOM tipo_evento evento_intensidade MES ANO ANO_MES
0   DJF  1950  24.72 -1.53     LA_NINA            3_FORTE  01  50   50_01
1   JFM  1950  25.17 -1.34     LA_NINA         2_MODERADO  02  50   50_02
2   FMA  1950  25.75 -1.16     LA_NINA         2_MODERADO  03  50   50_03
3   MAM  1950  26.12 -1.18     LA_NINA         2_MODERADO  04  50   50_04
4   AMJ  1950  26.32 -1.07     LA_NINA         2_MODERADO  05  50   50_05
5   MJJ  1950  26.31 -0.85     LA_NINA            1_FRACO  06  50   50_06
6   JJA  1950  26.21 -0.54     LA_NINA            1_FRACO  07  50   50_07
7   JAS  1950  25.96 -0.42      NEUTRO           0_NEUTRO  08  50   50_08
8   ASO  1950  25.76 -0.39      NEUTRO           0_NEUTRO  09  50   50_09
9   SON  1950  25.63 -0.44      NEUTRO           0_NEUTRO  10  50   50_10
10  OND  1950  25.48 -0.60     LA_NINA            1_FRACO  11  50   50_11
11  NDJ  1950  25.34 -0.80     LA_NINA            1_FRACO  12  50   50_12
Tabela 

In [27]:
import pandas as pd
import oracledb

# 1. Comando SQL para selecionar todos os dados da tabela
sql_consulta = "SELECT * FROM DADOS_CLIMA_GS_2026_S1"

try:
    # 2. Estabelecer a conexão com o banco de dados
    conn = oracledb.connect(
        user='rm567787',
        password='281083',
        dsn='oracle.fiap.com.br:1521/ORCL'
    )
    cursor = conn.cursor()

    # 3. Executar a consulta e buscar os dados e metadados das colunas
    cursor.execute(sql_consulta)
    dados = cursor.fetchall()

    # Extrai o nome exato de cada coluna do banco de dados
    colunas = [desc[0] for desc in cursor.description]

    # 4. Criar o DataFrame do Pandas de forma nativa e sem avisos
    df_clima = pd.DataFrame(dados, columns=colunas)

    print(f"Sucesso! {len(df_clima)} linhas importadas para o DataFrame 'df_clima'.")
except Exception as e:
    print(f"Erro ao importar dados: {e}")

finally:
    # 5. Garantir o fechamento correto de tudo
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn:
        conn.close()

Sucesso! 172221 linhas importadas para o DataFrame 'df_clima'.


In [28]:
df_clima['CHAVE'] = df_clima['COD'].astype(str) + '_' + df_clima['ANO_MES'].astype(str)
df_clima['MES_NUM'] = df_clima['ANO_MES'].astype(str).str[-2:].astype(int)
df_clima = df_clima.drop_duplicates(subset=['CHAVE'], keep='first').reset_index(drop=True)
df_clima['ANO'] = df_clima['ANO_MES'].astype(str).str[:2]

df_clima['CHAVE_ESTACAO'] = df_clima['COD'].astype(str) + '_' + df_clima['ESTACAO'].astype(str)

anos_desejados = ['24', '25', '26']

df_filtrado_anos = df_clima[df_clima['ANO'].isin(anos_desejados)]

ESTACAO_ATUAL = df_filtrado_anos[['CHAVE_ESTACAO']].drop_duplicates().reset_index(drop=True)

df_clima = pd.merge(
    ESTACAO_ATUAL,
    df_clima,
    left_on='CHAVE_ESTACAO',
    right_on='CHAVE_ESTACAO',
    how='inner')

print(f"Quantidade de linhas em df_clima após o merge (apenas coincidentes): {len(df_clima)}")

ESTACAO_ATUAL = df_filtrado_anos[['CHAVE_ESTACAO']].drop_duplicates().reset_index(drop=True)

print(f"Quantidade de linhas em ESTACAO_ATUAL após o merge (apenas coincidentes): {len(ESTACAO_ATUAL)}")

Quantidade de linhas em df_clima após o merge (apenas coincidentes): 110438
Quantidade de linhas em ESTACAO_ATUAL após o merge (apenas coincidentes): 599


In [35]:
import pandas as pd
import numpy as np
import oracledb

# ==============================================================================
# 1. EXTRAÇÃO E SANITIZAÇÃO GEOGRÁFICA (PANDAS)
# ==============================================================================
colunas_cadastro = [
    'REGIAO', 'UF', 'ESTACAO', 'COD', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'DATA_FUNDACAO', 'CHAVE_ESTACAO'
]

df_cadastro_estacoes = (
    df_clima[colunas_cadastro]
    .drop_duplicates(subset=['COD'], keep='first')
    .reset_index(drop=True)
    .copy()
)

df_cadastro_estacoes['LATITUDE'] = df_cadastro_estacoes['LATITUDE'].astype(str).str.replace(',', '.', regex=False)
df_cadastro_estacoes['LONGITUDE'] = df_cadastro_estacoes['LONGITUDE'].astype(str).str.replace(',', '.', regex=False)
df_cadastro_estacoes['ALTITUDE'] = df_cadastro_estacoes['ALTITUDE'].astype(str).str.replace(',', '.', regex=False)

df_cadastro_estacoes['LATITUDE'] = pd.to_numeric(df_cadastro_estacoes['LATITUDE'], errors='coerce')
df_cadastro_estacoes['LONGITUDE'] = pd.to_numeric(df_cadastro_estacoes['LONGITUDE'], errors='coerce')
df_cadastro_estacoes['ALTITUDE'] = pd.to_numeric(df_cadastro_estacoes['ALTITUDE'], errors='coerce')

df_cadastro_estacoes['dist_equador'] = df_cadastro_estacoes['LATITUDE'].abs() * 111.12

print(f"Quantidade de estações únicas encontradas: {len(df_cadastro_estacoes)}")

# ==============================================================================
# 2. PREPARAÇÃO DOS DADOS PARA O BANCO (GARANTIA DE TIPOS NATIVOS ORACLE)
# ==============================================================================
colunas_cadastro_sql = [
    'REGIAO', 'UF', 'ESTACAO', 'COD', 'LATITUDE', 
    'LONGITUDE', 'ALTITUDE', 'DATA_FUNDACAO', 'CHAVE_ESTACAO', 'dist_equador'
]
df_cadastro_insert = df_cadastro_estacoes[colunas_cadastro_sql].copy()

# CORREÇÃO DO AVISO: format='mixed' lida com múltiplos padrões de data sem warnings
df_cadastro_insert['DATA_FUNDACAO'] = pd.to_datetime(df_cadastro_insert['DATA_FUNDACAO'], errors='coerce', format='mixed')

df_cadastro_insert['LATITUDE'] = df_cadastro_insert['LATITUDE'].astype(float).where(df_cadastro_insert['LATITUDE'].notnull(), None)
df_cadastro_insert['LONGITUDE'] = df_cadastro_insert['LONGITUDE'].astype(float).where(df_cadastro_insert['LONGITUDE'].notnull(), None)
df_cadastro_insert['ALTITUDE'] = df_cadastro_insert['ALTITUDE'].astype(float).where(df_cadastro_insert['ALTITUDE'].notnull(), None)

dados_estacoes_inserir = []
for row in df_cadastro_insert.itertuples(index=False, name=None):
    linha_nativa = tuple(
        None if pd.isna(val) else (val.to_pydatetime() if isinstance(val, pd.Timestamp) else val)
        for val in row
    )
    dados_estacoes_inserir.append(linha_nativa)

# ==============================================================================
# 3. CONEXÃO, CRIAÇÃO DINÂMICA E EXECUÇÃO NO ORACLE
# ==============================================================================

# Script para criar a tabela caso ela não exista no servidor (Sem ESQUEMA.)
sql_criar_tabela = """
CREATE TABLE df_cadastro_estacoes (
    REGIAO VARCHAR2(50),
    UF VARCHAR2(10),
    ESTACAO VARCHAR2(150),
    COD VARCHAR2(50),
    LATITUDE NUMBER,
    LONGITUDE NUMBER,
    ALTITUDE NUMBER,
    DATA_FUNDACAO DATE,
    CHAVE_ESTACAO VARCHAR2(100),
    dist_equador NUMBER
)
"""

sql_limpeza = "TRUNCATE TABLE df_cadastro_estacoes"

sql_insercao = """
INSERT INTO df_cadastro_estacoes (
    REGIAO, UF, ESTACAO, COD, LATITUDE, LONGITUDE, ALTITUDE, DATA_FUNDACAO, CHAVE_ESTACAO, dist_equador
) VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9, :10)
"""

conn = None
cursor = None

try:
    conn = oracledb.connect(
        user='rm567787',
        password='281083',
        dsn='oracle.fiap.com.br:1521/ORCL'
    )
    cursor = conn.cursor()
    
    # --- ETAPA INTEGRADA: TESTE E CRIAÇÃO DA TABELA ---
    try:
        cursor.execute("SELECT 1 FROM df_cadastro_estacoes WHERE ROWNUM = 1")
        print("-> A tabela 'df_cadastro_estacoes' já existe no servidor.")
    except oracledb.DatabaseError as e:
        error_obj, = e.args
        if error_obj.code == 942:  # Código de erro para "Table or view does not exist"
            print("-> Tabela não encontrada. Executando script de criação para 'df_cadastro_estacoes'...")
            cursor.execute(sql_criar_tabela)
            print("-> Tabela criada com sucesso!")
        else:
            raise e  # Caso seja outro erro do Oracle, repassa a exceção
            
    # 1. Efetua a limpeza prévia da tabela de forma segura
    cursor.execute(sql_limpeza)
    print("-> Tabela limpa com sucesso via TRUNCATE.")
    
    # 2. Executa a inserção em massa utilizando Bulk Insert de alta performance
    cursor.executemany(sql_insercao, dados_estacoes_inserir)
    conn.commit()
    print(f"✅ Sucesso! {len(dados_estacoes_inserir)} linhas inseridas na tabela 'df_cadastro_estacoes'.")

except Exception as e:
    print(f"❌ Erro detetado no processo de carga Oracle: {e}")
    if conn:
        conn.rollback()
        print("-> Rollback efetuado para manter a integridade do banco de dados.")

finally:
    if cursor and getattr(cursor, '_impl', None) is not None:
        try:
            cursor.close()
            print("Cursor fechado com segurança.")
        except Exception:
            pass
            
    if conn:
        try:
            conn.close()
            print("Conexão com o Oracle encerrada.")
        except Exception:
            pass






Quantidade de estações únicas encontradas: 597
-> A tabela 'df_cadastro_estacoes' já existe no servidor.
-> Tabela limpa com sucesso via TRUNCATE.
✅ Sucesso! 597 linhas inseridas na tabela 'df_cadastro_estacoes'.
Cursor fechado com segurança.
Conexão com o Oracle encerrada.


In [33]:
import pandas as pd
import numpy as np
import oracledb

# ==============================================================================
# 1. GERAÇÃO E FILTRAGEM DOS DATAFRAMES HISTÓRICOS (PANDAS)
# ==============================================================================
HISTORICO_CHUVA = df_clima[(df_clima['PRECIPITACAO_MM'] >= 0)][['COD', 'PRECIPITACAO_MM', 'ANO_MES', 'MES_NUM']].copy()

HISTORICO_TEMPERATURA = df_clima[
    (df_clima['PRECIPITACAO_MM'] >= 0) & 
    (df_clima['TEMPERATURA_C'] >= -10) & 
    (df_clima['VENTO_RAJADA_MAX'] >= 0)
][['COD', 'TEMPERATURA_C', 'ANO_MES', 'MES_NUM']].copy()

HISTORICO_UMIDADE = df_clima[
    (df_clima['PRECIPITACAO_MM'] >= 0) & 
    (df_clima['UMIDADE_RELATIVA'] >= 0) & 
    (df_clima['VENTO_RAJADA_MAX'] >= 0) & 
    (df_clima['UMIDADE_MAX'] >= 0)
][['COD', 'UMIDADE_RELATIVA', 'ANO_MES', 'MES_NUM']].copy()

HISTORICO_VENTOS = df_clima[
    (df_clima['VENTO_VELOCIDADE'] >= 0) & 
    (df_clima['VENTO_VELOCIDADE'] != 0)
][['COD', 'VENTO_VELOCIDADE', 'ANO_MES', 'MES_NUM']].copy()

# ==============================================================================
# 2. CONFIGURAÇÃO DOS COMANDOS SQL E MAPEAMENTO DE COLUNAS
# ==============================================================================
ESQUEMA = "RM567787"  # Define o esquema para evitar erros de tabelas não encontradas

tabelas_config = {
    "HISTORICO_CHUVA": {
        "df": HISTORICO_CHUVA,
        "colunas": ['COD', 'PRECIPITACAO_MM', 'ANO_MES', 'MES_NUM'],
        "sql_create": f"""
            CREATE TABLE {ESQUEMA}.HISTORICO_CHUVA (
                COD VARCHAR2(50),
                PRECIPITACAO_MM NUMBER,
                ANO_MES VARCHAR2(10),
                MES_NUM NUMBER
            )
        """,
        "sql_truncate": f"TRUNCATE TABLE {ESQUEMA}.HISTORICO_CHUVA",
        "sql_insert": f"INSERT INTO {ESQUEMA}.HISTORICO_CHUVA (COD, PRECIPITACAO_MM, ANO_MES, MES_NUM) VALUES (:1, :2, :3, :4)"
    },
    "HISTORICO_TEMPERATURA": {
        "df": HISTORICO_TEMPERATURA,
        "colunas": ['COD', 'TEMPERATURA_C', 'ANO_MES', 'MES_NUM'],
        "sql_create": f"""
            CREATE TABLE {ESQUEMA}.HISTORICO_TEMPERATURA (
                COD VARCHAR2(50),
                TEMPERATURA_C NUMBER,
                ANO_MES VARCHAR2(10),
                MES_NUM NUMBER
            )
        """,
        "sql_truncate": f"TRUNCATE TABLE {ESQUEMA}.HISTORICO_TEMPERATURA",
        "sql_insert": f"INSERT INTO {ESQUEMA}.HISTORICO_TEMPERATURA (COD, TEMPERATURA_C, ANO_MES, MES_NUM) VALUES (:1, :2, :3, :4)"
    },
    "HISTORICO_UMIDADE": {
        "df": HISTORICO_UMIDADE,
        "colunas": ['COD', 'UMIDADE_RELATIVA', 'ANO_MES', 'MES_NUM'],
        "sql_create": f"""
            CREATE TABLE {ESQUEMA}.HISTORICO_UMIDADE (
                COD VARCHAR2(50),
                UMIDADE_RELATIVA NUMBER,
                ANO_MES VARCHAR2(10),
                MES_NUM NUMBER
            )
        """,
        "sql_truncate": f"TRUNCATE TABLE {ESQUEMA}.HISTORICO_UMIDADE",
        "sql_insert": f"INSERT INTO {ESQUEMA}.HISTORICO_UMIDADE (COD, UMIDADE_RELATIVA, ANO_MES, MES_NUM) VALUES (:1, :2, :3, :4)"
    },
    "HISTORICO_VENTOS": {
        "df": HISTORICO_VENTOS,
        "colunas": ['COD', 'VENTO_VELOCIDADE', 'ANO_MES', 'MES_NUM'],
        "sql_create": f"""
            CREATE TABLE {ESQUEMA}.HISTORICO_VENTOS (
                COD VARCHAR2(50),
                VENTO_VELOCIDADE NUMBER,
                ANO_MES VARCHAR2(10),
                MES_NUM NUMBER
            )
        """,
        "sql_truncate": f"TRUNCATE TABLE {ESQUEMA}.HISTORICO_VENTOS",
        "sql_insert": f"INSERT INTO {ESQUEMA}.HISTORICO_VENTOS (COD, VENTO_VELOCIDADE, ANO_MES, MES_NUM) VALUES (:1, :2, :3, :4)"
    }
}

# ==============================================================================
# 3. EXECUÇÃO DA CRIAÇÃO E CARGA EM LOTE NO SERVIDOR ORACLE
# ==============================================================================
conn = None
cursor = None

try:
    # Conexão utilizando o DSN correto para o oracledb
    conn = oracledb.connect(
        user='rm567787',
        password='281083',
        dsn='oracle.fiap.com.br:1521/ORCL'
    )
    cursor = conn.cursor()
    
    for nome_tabela, config in tabelas_config.items():
        print(f"\nIniciando processamento da tabela: {nome_tabela}...")
        
        # --- VERIFICAÇÃO AUTOMÁTICA DE EXISTÊNCIA DA TABELA ---
        try:
            cursor.execute(f"SELECT 1 FROM {ESQUEMA}.{nome_tabela} WHERE ROWNUM = 1")
            print(f"-> A tabela '{nome_tabela}' já existe.")
        except oracledb.DatabaseError as e:
            error_obj, = e.args
            if error_obj.code == 942:  # ORA-00942: table or view does not exist
                print(f"-> A tabela '{nome_tabela}' não foi encontrada. Criando estrutura...")
                cursor.execute(config["sql_create"])
                print(f"-> Tabela '{nome_tabela}' criada com sucesso.")
            else:
                raise e
        
        # Garante a ordem exata das colunas extraídas do DataFrame
        df_insert = config["df"][config["colunas"]].copy()
        
        # CORREÇÃO DE TIPOS: Extrai os dados forçando tipos puros (int, float, str, None)
        dados_inserir = []
        for row in df_insert.itertuples(index=False, name=None):
            linha_nativa = tuple(
                None if pd.isna(val) 
                else (int(val) if isinstance(val, (np.integer, int)) 
                      else (float(val) if isinstance(val, (np.floating, float)) else val))
                for val in row
            )
            dados_inserir.append(linha_nativa)
            
        # 1. Limpeza física da tabela no banco
        cursor.execute(config["sql_truncate"])
        print(f"-> Dados antigos limpos via TRUNCATE.")
        
        # 2. Inserção em massa de alta performance (Bulk Insert)
        if dados_inserir:
            cursor.executemany(config["sql_insert"], dados_inserir)
            print(f"-> Sucesso! {len(dados_inserir)} registros preparados para '{nome_tabela}'.")
        else:
            print(f"-> Aviso: Nenhum registro encontrado para salvar em '{nome_tabela}'.")
            
    # Confirmação atómica: salva todas as tabelas juntas no banco de dados
    conn.commit()
    print("\n✅ Todas as tabelas de histórico foram replicadas com sucesso no servidor!")

except Exception as e:
    print(f"\n❌ Erro fatal detetado no processo de carga: {e}")
    if conn:
        conn.rollback()
        print("-> Rollback executado: Nenhuma alteração foi guardada no banco.")

finally:
    # Fecho blindado do cursor contra o erro DPY-1006
    if cursor and getattr(cursor, '_impl', None) is not None:
        try:
            cursor.close()
            print("Cursor fechado com segurança.")
        except Exception:
            pass
            
    # Fecho seguro da ligação de rede
    if conn:
        try:
            conn.close()
            print("Conexão com o servidor Oracle encerrada.")
        except Exception:
            pass



Iniciando processamento da tabela: HISTORICO_CHUVA...
-> A tabela 'HISTORICO_CHUVA' não foi encontrada. Criando estrutura...
-> Tabela 'HISTORICO_CHUVA' criada com sucesso.
-> Dados antigos limpos via TRUNCATE.
-> Sucesso! 84730 registros preparados para 'HISTORICO_CHUVA'.

Iniciando processamento da tabela: HISTORICO_TEMPERATURA...
-> A tabela 'HISTORICO_TEMPERATURA' não foi encontrada. Criando estrutura...
-> Tabela 'HISTORICO_TEMPERATURA' criada com sucesso.
-> Dados antigos limpos via TRUNCATE.
-> Sucesso! 72299 registros preparados para 'HISTORICO_TEMPERATURA'.

Iniciando processamento da tabela: HISTORICO_UMIDADE...
-> A tabela 'HISTORICO_UMIDADE' não foi encontrada. Criando estrutura...
-> Tabela 'HISTORICO_UMIDADE' criada com sucesso.
-> Dados antigos limpos via TRUNCATE.
-> Sucesso! 69977 registros preparados para 'HISTORICO_UMIDADE'.

Iniciando processamento da tabela: HISTORICO_VENTOS...
-> A tabela 'HISTORICO_VENTOS' não foi encontrada. Criando estrutura...
-> Tabela 'HIST